# Day 3: Pytho Folder Structure with Docker

## Goal

Move the FastAPI app from a flat script or notebook into a standard project folder structure and containerize it.

By the end of this notebook you will have:

- an organized Python backend structure
- a `Dockerfile` to containerize the API
- an understanding of how to build and run the container
- a runnable Docker container serving the API

This prepares the app for real-world deployment.

## What is Docker?

Docker is a platform that uses OS-level virtualization to deliver software in packages called **containers**. A container holds everything needed for an application to run (code, runtime, system tools, system libraries, and settings) ensuring it runs consistently regardless of the host environment.

## Importance of Docker

1. **Consistency:** "It works on my machine" becomes a thing of the past. The container behaves exactly the same in development, testing, and production.
2. **Portability:** Containers can run on any OS that supports Docker (Linux, Windows, macOS) and on any cloud provider.
3. **Isolation:** Dependencies and processes are isolated from the host machine and other containers.

## Why Adding Docker Makes Python Reliable

Python environments are notoriously tricky to manage due to system-level library dependencies, varying Python versions, and virtual environment conflicts. 

By putting your Python API in a Docker container:
- You lock in the exact Python version (e.g., `python:3.12-slim`).
- You lock in binary system dependencies (like C++ build tools for AI libraries) without cluttering the host machine.
- You ensure the production server runs the exact identical environment that you developed and tested locally.

## Step 1: Define the Folder Structure

A standard pattern for Python APIs puts application code in an `app` directory, keeps dependencies explicitly listed, and isolates deployment logic in a `Dockerfile`.

We will create the following structure inside an `api_project` folder:

```
api_project/
├── app/
│   ├── __init__.py
│   ├── main.py
│   └── api/
│       ├── __init__.py
│       └── endpoints.py
├── requirements.txt
└── Dockerfile
```

Let's create those directories now.

In [ ]:
import os

base_dir = "api_project"
os.makedirs(f"{base_dir}/app/api", exist_ok=True)

with open(f"{base_dir}/app/__init__.py", "w") as f:
    f.write("")

with open(f"{base_dir}/app/api/__init__.py", "w") as f:
    f.write("")

print("Folder structure initialized.")

## Step 2: Write Application Code

We use the `%%writefile` magic to generate the Python files. 

First, the route definitions in `endpoints.py`.

In [ ]:
%%writefile api_project/app/api/endpoints.py
from fastapi import APIRouter

router = APIRouter()

@router.get("/health")
def health_check():
    return {"status": "ok", "containerized": True}

@router.get("/query")
def query_placeholder():
    return {"message": "This endpoint will connect to LLMs in future steps."}


Next, wire them into the main FastAPI application in `main.py`.

In [ ]:
%%writefile api_project/app/main.py
from fastapi import FastAPI
from app.api.endpoints import router

app = FastAPI(title="Day 3 Container API")
app.include_router(router, prefix="/api/v1")

@app.get("/")
def root():
    return {"message": "Welcome to the Containerized API"}


## Step 3: Dependencies and Dockerfile

For this standalone container, we'll list explicit requirements. In production, tools like `uv` or `poetry` can export these.

In [ ]:
%%writefile api_project/requirements.txt
fastapi>=0.135.1
uvicorn[standard]>=0.42.0


Now we define the `Dockerfile`. It starts from a slim Python 3.12 image, installs requirements, copies our `app` folder, and exposes port 8000.

In [ ]:
%%writefile api_project/Dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    WORKDIR=/src

WORKDIR ${WORKDIR}

COPY requirements.txt .
RUN pip install --no-cache-dir -U pip setuptools && \
    pip install --no-cache-dir -r requirements.txt

COPY ./app ./app

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]


## Step 4: Build functionality

We can build the Docker image directly from this notebook using shell commands (`!`).

*(Note: You must have a Docker daemon running on your host machine to run the following cells. If you do not, you can review the commands here and run them later.)*

In [ ]:
!cd api_project && docker build -t day3-api:latest .
print("Docker image built.")

## Step 5: Run & Test the Container

We can start the container in detached mode (`-d`), mapping host port 8000 to container port 8000.

In [ ]:
# Start the container
!docker run -d -p 8000:8000 --name my_day3_api day3-api:latest

import time
time.sleep(2) # Wait a moment for the server to spin up inside the container
print("Container started on port 8000.")

Now let's query the running container.

In [ ]:
import urllib.request
import json

try:
    req = urllib.request.Request("http://127.0.0.1:8000/api/v1/health")
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode())
        print("API Response:", data)
except Exception as e:
    print("Failed to reach API:", e)


Finally, clean up by stopping and removing the container so it doesn't block port 8000 later.

In [ ]:
!docker stop my_day3_api
!docker rm my_day3_api
print("Cleanup complete.")

## Day 3 Recap

You have successfully:

- Restructured a basic Python app into a standard `app/` structure with separate routers.
- Written a `Dockerfile` that packages Python dependencies and source code.
- Built a standard runnable container (`day3-api:latest`).
- Validated the application running fully isolated inside Docker.

This stable deployment pattern will be used when we deploy more complex AI endpoints in subsequent days.